# 01 - ETL to Neo4j

Load the Aircraft Digital Twin topology into Neo4j from the committed `data/` directory, then verify it with Cypher. This notebook uses the **Neo4j Python driver** (not the Spark Connector), so it runs anywhere the `neo4j` package is installed.

## What you'll learn
- Read the committed CSVs through the `DATA_SOURCE` switch (GitHub, local, or volume)
- Load nodes and relationships with `UNWIND` batch writes
- Verify node and relationship counts with Cypher

## Prerequisites
- A Neo4j instance and its credentials (Aura free tier is enough)
- `pip install neo4j`

This is the first notebook in the series. Run it before 02-05.

## Section 1: Configuration

Enter your Neo4j credentials and pick where to read the data from.

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

NEO4J_URI = ""           # e.g. "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""       # your Neo4j password

# Where to read data/ from:
#   "github"  -> raw files from the public repo (default, zero setup)
#   "local"   -> a local clone (./data)
#   "volume"  -> a Unity Catalog volume you have populated
DATA_SOURCE = "github"

# Clear the database before loading (safe to leave True for a clean run)
CLEAR_DATABASE = True

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: enter your Neo4j credentials above before running!")
else:
    print(f"Configuration ready - {NEO4J_URI} (DATA_SOURCE={DATA_SOURCE})")

In [ ]:
%pip install neo4j

In [ ]:
from data_loader import Neo4jConnection, load_csv

neo4j = Neo4jConnection(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD).verify()
driver = neo4j.driver

## Section 2: Loader helpers

Two helpers load every node and relationship file. They read the committed CSVs through `load_csv`, then write them to Neo4j with `UNWIND`.

The CSV headers use the Neo4j import format (`:ID(Aircraft)`, `:START_ID(Flight)`, `:END_ID(Airport)`). The helpers detect those columns by their label, so the same code handles every file. Each helper prints `created/total`: on a clean run the two match; a shortfall means some relationship endpoints were missing.

In [ ]:
def load_nodes(filename, label, id_prop):
    """Load a node CSV. The :ID(...) column becomes id_prop; other columns become properties."""
    rows = load_csv(filename, DATA_SOURCE)
    id_col = next(k for k in rows[0] if k.startswith(":ID("))
    for row in rows:
        row[id_prop] = row.pop(id_col)
    _, summary, _ = driver.execute_query(
        f"UNWIND $rows AS row "
        f"MERGE (n:{label} {{{id_prop}: row.{id_prop}}}) "
        f"SET n += row",
        rows=rows,
    )
    created = summary.counters.nodes_created
    print(f"  {created:>4}/{len(rows):<4} :{label}")
    return len(rows)


def load_rels(filename, rel_type, src_label, src_prop, src_id_label,
              tgt_label, tgt_prop, tgt_id_label):
    """Load a relationship CSV. Columns are matched by their (Label) marker, not by START/END,
    so reversed files (REMOVED_COMPONENT) load in the intended direction."""
    rows = load_csv(filename, DATA_SOURCE)
    src_col = next(k for k in rows[0] if k.endswith(f"({src_id_label})"))
    tgt_col = next(k for k in rows[0] if k.endswith(f"({tgt_id_label})"))
    pairs = [{"s": r[src_col], "t": r[tgt_col]} for r in rows]
    _, summary, _ = driver.execute_query(
        f"UNWIND $pairs AS p "
        f"MATCH (s:{src_label} {{{src_prop}: p.s}}) "
        f"MATCH (t:{tgt_label} {{{tgt_prop}: p.t}}) "
        f"MERGE (s)-[:{rel_type}]->(t)",
        pairs=pairs,
    )
    created = summary.counters.relationships_created
    print(f"  {created:>4}/{len(pairs):<4} {rel_type}")
    return len(pairs)

## Section 3: Clear and constrain

Optionally clear the database, then create uniqueness constraints. Constraints make the `MERGE` writes correct and fast.

In [ ]:
if CLEAR_DATABASE:
    print("Clearing database...")
    while True:
        records, _, _ = driver.execute_query(
            "MATCH (n) WITH n LIMIT 10000 DETACH DELETE n RETURN count(*) AS deleted"
        )
        if records[0]["deleted"] == 0:
            break
    print("  database cleared")

CONSTRAINTS = [
    ("Aircraft", "aircraft_id"), ("System", "system_id"), ("Component", "component_id"),
    ("Sensor", "sensor_id"), ("Airport", "airport_id"), ("Flight", "flight_id"),
    ("Delay", "delay_id"), ("MaintenanceEvent", "event_id"), ("Removal", "removal_id"),
]
for label, prop in CONSTRAINTS:
    driver.execute_query(
        f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.{prop} IS UNIQUE"
    )
print(f"Created {len(CONSTRAINTS)} uniqueness constraints")

## Section 4: Load nodes

Nine node types: the `Aircraft -> System -> Component -> Sensor` topology, plus `Flight`, `Airport`, `Delay`, `MaintenanceEvent`, and `Removal`.

In [ ]:
NODE_LOADS = [
    ("nodes_aircraft.csv",    "Aircraft",         "aircraft_id"),
    ("nodes_systems.csv",     "System",           "system_id"),
    ("nodes_components.csv",  "Component",        "component_id"),
    ("nodes_sensors.csv",     "Sensor",           "sensor_id"),
    ("nodes_airports.csv",    "Airport",          "airport_id"),
    ("nodes_flights.csv",     "Flight",           "flight_id"),
    ("nodes_delays.csv",      "Delay",            "delay_id"),
    ("nodes_maintenance.csv", "MaintenanceEvent", "event_id"),
    ("nodes_removals.csv",    "Removal",          "removal_id"),
]

print("Loading nodes (created/total)...")
total_nodes = sum(load_nodes(*args) for args in NODE_LOADS)
print(f"Total: {total_nodes} node rows processed")

## Section 5: Load relationships

Twelve relationship types connect the topology, the flight network, and the maintenance chain.

In [ ]:
REL_LOADS = [
    # filename, rel_type, src_label, src_prop, src_id_label, tgt_label, tgt_prop, tgt_id_label
    ("rels_aircraft_system.csv",   "HAS_SYSTEM",        "Aircraft", "aircraft_id", "Aircraft",        "System",           "system_id",    "System"),
    ("rels_system_component.csv",  "HAS_COMPONENT",     "System",   "system_id",   "System",          "Component",        "component_id", "Component"),
    ("rels_system_sensor.csv",     "HAS_SENSOR",        "System",   "system_id",   "System",          "Sensor",           "sensor_id",    "Sensor"),
    ("rels_component_event.csv",   "HAS_EVENT",         "Component","component_id","Component",        "MaintenanceEvent", "event_id",     "MaintenanceEvent"),
    ("rels_aircraft_flight.csv",   "OPERATES_FLIGHT",   "Aircraft", "aircraft_id", "Aircraft",        "Flight",           "flight_id",    "Flight"),
    ("rels_flight_departure.csv",  "DEPARTS_FROM",      "Flight",   "flight_id",   "Flight",          "Airport",          "airport_id",   "Airport"),
    ("rels_flight_arrival.csv",    "ARRIVES_AT",        "Flight",   "flight_id",   "Flight",          "Airport",          "airport_id",   "Airport"),
    ("rels_flight_delay.csv",      "HAS_DELAY",         "Flight",   "flight_id",   "Flight",          "Delay",            "delay_id",     "Delay"),
    ("rels_event_system.csv",      "AFFECTS_SYSTEM",    "MaintenanceEvent", "event_id", "MaintenanceEvent", "System",      "system_id",    "System"),
    ("rels_event_aircraft.csv",    "AFFECTS_AIRCRAFT",  "MaintenanceEvent", "event_id", "MaintenanceEvent", "Aircraft",    "aircraft_id",  "Aircraft"),
    ("rels_aircraft_removal.csv",  "HAS_REMOVAL",       "Aircraft", "aircraft_id", "Aircraft",        "Removal",          "removal_id",   "RemovalEvent"),
    ("rels_component_removal.csv", "REMOVED_COMPONENT", "Removal",  "removal_id",  "RemovalEvent",    "Component",         "component_id", "Component"),
]

print("Loading relationships (created/total)...")
total_rels = sum(load_rels(*args) for args in REL_LOADS)
print(f"Total: {total_rels} relationship rows processed")

## Section 6: Verify

Count nodes by label and relationships by type. This is the authoritative check on what landed in the database.

In [ ]:
print("Node counts by label:")
records, _, _ = driver.execute_query("""
    MATCH (n)
    UNWIND labels(n) AS label
    RETURN label, count(*) AS count
    ORDER BY count DESC
""")
for r in records:
    print(f"  {r['label']:<18} {r['count']:>5}")

print("\nRelationship counts by type:")
records, _, _ = driver.execute_query("""
    MATCH ()-[r]->()
    RETURN type(r) AS rel_type, count(*) AS count
    ORDER BY count DESC
""")
for r in records:
    print(f"  {r['rel_type']:<18} {r['count']:>5}")

## Next steps

The aircraft topology is loaded. From here:

- **02_gds_louvain_maintenance** (optional) - run Graph Data Science over the maintenance chain
- **03_data_and_embeddings** - load the maintenance manuals, chunk, embed, and index them

Try a query in your Neo4j console to see one aircraft's hierarchy:

```cypher
MATCH (a:Aircraft)-[:HAS_SYSTEM]->(s:System)-[:HAS_COMPONENT]->(c:Component)
RETURN a, s, c
LIMIT 50
```

In [ ]:
neo4j.close()